# Signal Research

Test momentum and mean reversion signals, compare performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Load Data

In [ ]:
from crypto_quant.data.loaders import load_canonical_dataset

data = load_canonical_dataset()
print(f"Loaded data: {data.shape}")

## Generate Signals

In [ ]:
from crypto_quant.signals.momentum import MomentumSignal
from crypto_quant.signals.mean_reversion import MeanReversionSignal
from crypto_quant.signals.composite import CompositeSignal

# Momentum signal
momentum = MomentumSignal(lookback=5, threshold=0.001)
momentum_signal = momentum.generate(data)

# Mean reversion signal
mean_reversion = MeanReversionSignal(window=20, threshold=1.5)
mr_signal = mean_reversion.generate(data)

# Composite signal
composite = CompositeSignal(
    signals=[momentum, mean_reversion],
    weights={'Momentum_5': 0.5, 'MeanReversion_20': 0.5}
)
composite_signal = composite.generate(data)

print(f"Momentum signal: {momentum_signal.value_counts()}")
print(f"Mean Reversion signal: {mr_signal.value_counts()}")
print(f"Composite signal: {composite_signal.value_counts()}")

## Signal Analysis

In [ ]:
from crypto_quant.research.diagnostics import ResearchDiagnostics

print("=== Momentum Signal ===")
mom_dist = ResearchDiagnostics.analyze_signal_distribution(momentum_signal)
for k, v in mom_dist.items():
    print(f"{k}: {v}")

print("\n=== Mean Reversion Signal ===")
mr_dist = ResearchDiagnostics.analyze_signal_distribution(mr_signal)
for k, v in mr_dist.items():
    print(f"{k}: {v}")

print("\n=== Composite Signal ===")
comp_dist = ResearchDiagnostics.analyze_signal_distribution(composite_signal)
for k, v in comp_dist.items():
    print(f"{k}: {v}")

## Plot Signals

In [ ]:
# Plot signals overlaid on price
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

# Price
axes[0].plot(data.index, data['close'], linewidth=0.5)
axes[0].set_title('BTCUSDT Close Price')
axes[0].set_ylabel('Price')
axes[0].grid(True)

# Momentum signal
axes[1].scatter(data.index[momentum_signal == 1], data['close'][momentum_signal == 1], 
                color='green', s=1, alpha=0.5, label='Long')
axes[1].scatter(data.index[momentum_signal == -1], data['close'][momentum_signal == -1], 
                color='red', s=1, alpha=0.5, label='Short')
axes[1].set_title('Momentum Signal')
axes[1].set_ylabel('Signal')
axes[1].legend()
axes[1].grid(True)

# Mean reversion signal
axes[2].scatter(data.index[mr_signal == 1], data['close'][mr_signal == 1], 
                color='green', s=1, alpha=0.5, label='Long')
axes[2].scatter(data.index[mr_signal == -1], data['close'][mr_signal == -1], 
                color='red', s=1, alpha=0.5, label='Short')
axes[2].set_title('Mean Reversion Signal')
axes[2].set_ylabel('Signal')
axes[2].legend()
axes[2].grid(True)

# Composite signal
axes[3].scatter(data.index[composite_signal == 1], data['close'][composite_signal == 1], 
                color='green', s=1, alpha=0.5, label='Long')
axes[3].scatter(data.index[composite_signal == -1], data['close'][composite_signal == -1], 
                color='red', s=1, alpha=0.5, label='Short')
axes[3].set_title('Composite Signal')
axes[3].set_ylabel('Signal')
axes[3].set_xlabel('Date')
axes[3].legend()
axes[3].grid(True)

plt.tight_layout()
plt.show()

## Simple Backtest Comparison

In [ ]:
# Simple buy-and-hold return
total_return_buyhold = (data['close'].iloc[-1] / data['close'].iloc[0]) - 1
print(f"Buy & Hold Total Return: {total_return_buyhold:.2%}")

# Compute PnL for each signal
returns = data['close'].pct_change()

for signal_name, signal in [('Momentum', momentum_signal), 
                            ('Mean Reversion', mr_signal), 
                            ('Composite', composite_signal)]:
    # Strategy PnL = signal * return
    strategy_pnl = signal * returns
    cumulative_pnl = (1 + strategy_pnl).cumprod()
    total_return = cumulative_pnl.iloc[-1] - 1
    
    print(f"\n{signal_name} Signal:")
    print(f"  Total Return: {total_return:.2%}")
    print(f"  Sharpe Ratio: {(strategy_pnl.mean() / strategy_pnl.std()) * np.sqrt(252*1440):.2f}")
    print(f"  Win Rate: {(strategy_pnl > 0).sum() / len(strategy_pnl):.2%}")